# Process Probe Drift Results

This notebook loads experiment outputs, builds tabular views for exploration, and plots per-turn relative alignment ($A_t$).

In [ ]:
from pathlib import Path
import pandas as pd

import sys
sys.path.append("/home/USER/dev/activation-drift")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

## Load CSV and JSONL Outputs

In [ ]:
from activation_drift.process_results import choose_latest_run, load_raw_results, load_config, get_tradeoffs_per_conversation, make_alignment_dfs, build_turns_df, print_data_summary

repo_root = Path('/home/USER/dev/activation-drift')

model = 'llama70B'

normalization_json = repo_root / f"probes/{model}/best_probe_calibration_train.json"

#RESULT_DIR = choose_latest_run(artifacts_root)
#RESULT_DIR = artifacts_root / "probe_experiment_ollama_8bit_demo_10seeds_systemprompt"
RESULT_DIR = Path(f"/home/USER/dev/activation-drift/results/{model}/caa_4")

USE_DELTA_WITH_BASELINE = False  # If True, subtract the baseline's A_t from the condition's A_t

In [ ]:

probe_scores_by_turn_df, normalization_stats, run_records = load_raw_results(RESULT_DIR, normalization_json=normalization_json)
config = load_config(RESULT_DIR)

tradeoff_by_conv, run_meta_by_key = get_tradeoffs_per_conversation(run_records)

alignment_by_turn_df, alignment_repeat_stats_df = make_alignment_dfs(probe_scores_by_turn_df, run_meta_by_key, normalization_stats, use_delta_with_baseline=USE_DELTA_WITH_BASELINE)
turns_df = build_turns_df(run_records, alignment_repeat_stats_df) # turn-level dataset for inspection

print_data_summary(config, run_records, probe_scores_by_turn_df, alignment_by_turn_df, alignment_repeat_stats_df)

## Compute D and TV

Use the unaggregated per-seed turns $A_t$ to compute $D$ and $TV$ for each seed run, then summarize their mean and std across seeds.

In [ ]:
from activation_drift.evaluation import compute_alignment_metrics

metric_df = compute_alignment_metrics(alignment_by_turn_df, use_delta_with_baseline=USE_DELTA_WITH_BASELINE)

## Plot results

In [ ]:
from activation_drift.plot_results import plot_D_TV, plot_specific_trajectory, plot_tradeoff_aggregate, inspect_conversation

### Inspect individual conversation and $A_t$ evolution over turns

In [ ]:
inspect_conversation(alignment_by_turn_df, run_records, use_delta_with_baseline=USE_DELTA_WITH_BASELINE)

### D and TV by Condition

In [ ]:
filter_by = "SELF-DIRECTION"  # choose among "POWER", "SELF-DIRECTION", or "" to not filter


plot_metrics = plot_D_TV(metric_df[metric_df["scenario_value_a"] == filter_by]) if filter_by else plot_D_TV(metric_df)

### Alignment trajectory $A_t$ for a specific condition 

Pick a conversation to inspect trajectories by condition.

In [ ]:
print("Available conversation IDs:", list(alignment_repeat_stats_df['conversation_id'].unique()))
print("Available conditions:", list(alignment_repeat_stats_df['condition'].unique()))

In [ ]:
selected_conversation = 'tradeoff1_career_change'

selected_conditions = ['baseline']

trajectories = plot_specific_trajectory(alignment_repeat_stats_df, selected_conversation, selected_conditions, seed=42, alignment_by_turn_df=alignment_by_turn_df)

In [ ]:
selected_conversation = 'tradeoff2_environmental_impact'

selected_conditions = ['model_va_user_vb', 'model_vb_user_va']

trajectories = plot_specific_trajectory(alignment_repeat_stats_df, selected_conversation, selected_conditions)

### Results aggregated over scenarios

Given a trade-off, This figure shows one per steering direction, aggregated over the conversation scenarios for that value trade-off. Select steering condition from [baseline, both, opposite, only_user].

In [ ]:
trajectories = plot_tradeoff_aggregate(alignment_repeat_stats_df, value_a="SELF-DIRECTION", value_b="SECURITY", condition="opposite") #for trade-off 1

In [ ]:
trajectories =  plot_tradeoff_aggregate(alignment_repeat_stats_df, value_a="POWER", value_b="UNIVERSALISM", condition="both") #for trade-off2

# Conversation quality

We measure conversation quality through coherence and repetition across turns.

In [ ]:
from pathlib import Path
import pandas as pd

import sys
sys.path.append("/home/USER/dev/activation-drift")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

model = 'qwen'

RESULT_DIR = Path(f"/home/USER/dev/activation-drift/results/{model}/systemprompt_power")

conversation_quality_pickle = RESULT_DIR / "conversation_quality.pkl"

In [ ]:
from activation_drift.utils import load_pickle

conversation_quality_results = load_pickle(conversation_quality_pickle)

In [ ]:
conversation_quality_results["summary"]["coherence"]